## Discovering Reddit communities
02807 Computational Tools for Data Science, Group 31

# 1. Data

In [ ]:
# Run this script to have the necessary packages
#pip install requests pandas

## 1.1 Importing packages and choosing data

In [2]:
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://arctic-shift.photon-reddit.com/api"

SUBREDDIT = "datascience"  # Change this; do not include r/

# Include September 1, exclude October 1, using UTC.
START = int(pd.Timestamp("2026-09-01", tz="UTC").timestamp())
END = int(pd.Timestamp("2026-10-01", tz="UTC").timestamp())

POST_FIELDS = [
    "id", "subreddit", "author", "created_utc",
    "title", "selftext", "score", "num_comments", "url"
]

COMMENT_FIELDS = [
    "id", "subreddit", "author", "created_utc",
    "body", "score", "link_id", "parent_id"
]

## 1.2 Fetching batches

In [3]:
def fetch_data(kind, fields):
    records = {}
    cursor = START - 1

    while True:
        response = requests.get(
            f"{BASE_URL}/{kind}/search",
            params={
                "subreddit": SUBREDDIT,
                "after": cursor,
                "before": END,
                "sort": "asc",
                "limit": 100,
                "fields": ",".join(fields),
            },
            timeout=60,
        )

        # Stop with an error if the request failed.
        response.raise_for_status()
        batch = response.json()["data"]

        if not batch:
            break

        # Keep only September records.
        # Using IDs as dictionary keys also removes duplicates.
        for record in batch:
            if START <= record["created_utc"] < END:
                records[record["id"]] = record

        print(f"{kind}: {len(records)} records collected")

        if len(batch) < 100:
            break

        # Overlap the final second to avoid skipping timestamp ties.
        next_cursor = max(r["created_utc"] for r in batch) - 1

        if next_cursor <= cursor:
            raise RuntimeError(
                "Pagination cannot advance: too many records share "
                "one timestamp. Use the monthly dump for this interval."
            )

        cursor = next_cursor
        time.sleep(1)  # Pause between requests.

    df = pd.DataFrame(records.values(), columns=fields)

    df["created_at"] = pd.to_datetime(
        df["created_utc"], unit="s", utc=True
    )

    return df.sort_values("created_utc").reset_index(drop=True)

## 1.3 Inspect data
The function requests up to 100 records at a time, stores them, then moves the date cursor forward. If a request fails, it stops instead of silently saving an incomplete result. For HTTP 429, wait before rerunning: that means the API has rate limited the requests.

### 1.3.1 Fetch posts

In [4]:
posts = fetch_data("posts", POST_FIELDS)

print(f"Posts collected: {len(posts)}")
display(posts.head())

posts: 71 records collected
Posts collected: 71


,id,subreddit,author,created_utc,title,selftext,score,num_comments,url,created_at
0,1w484pi,datascience,bobbe_,1788258953,Pulling data from access orchestrate to build ...,[removed],1,1,https://www.reddit.com/r/datascience/comments/...,2026-09-01 10:35:53+00:00
1,1w4c38i,datascience,DataScienceGuy_,1788269475,"At senior levels, where do you draw the line b...",I’ve been in data science/analytics for about ...,22,29,https://www.reddit.com/r/datascience/comments/...,2026-09-01 13:31:15+00:00
2,1w67e3p,datascience,Trick-Interaction396,1788442016,How are LLMs used in predictive modeling and a...,I am DS with 20 YOE but I've been managing lat...,54,37,https://www.reddit.com/r/datascience/comments/...,2026-09-03 13:26:56+00:00
3,1w6qtza,datascience,informatica6,1788486439,"How do you guys ""show your work""",I use a google sheet to list my major projects...,12,14,https://www.reddit.com/r/datascience/comments/...,2026-09-04 01:47:19+00:00
4,1w70acq,datascience,Repulsive-Look-9787,1788516835,Is sql important for a data analyst internship,[removed],1,1,https://www.reddit.com/r/datascience/comments/...,2026-09-04 10:13:55+00:00


### 1.3.2 Fetch comments

In [5]:
comments = fetch_data("comments", COMMENT_FIELDS)

print(f"Comments collected: {len(comments)}")
display(comments.head())

comments: 100 records collected
comments: 199 records collected
comments: 298 records collected
comments: 397 records collected
comments: 496 records collected
comments: 595 records collected
comments: 694 records collected
comments: 793 records collected
comments: 892 records collected
comments: 946 records collected
Comments collected: 946


,id,subreddit,author,created_utc,body,score,link_id,parent_id,created_at
0,p732qz6,datascience,[deleted],1788221176,[removed],1,t3_1w3klb5,t3_1w3klb5,2026-09-01 00:06:16+00:00
1,p732zzz,datascience,aaronswar43,1788221258,Had an offer from Amazon and I decided to decl...,1,t3_1w3klb5,t3_1w3klb5,2026-09-01 00:07:38+00:00
2,p734a57,datascience,Dependent-Aide-388,1788221677,I have worked at multiple FAANGs and I do regr...,1,t3_1w3klb5,t3_1w3klb5,2026-09-01 00:14:37+00:00
3,p734irn,datascience,ExecutiveFingerblast,1788221756,There's an old Eddie izzard bit about jfk in G...,2,t3_1w3n6p7,t1_p72u2fd,2026-09-01 00:15:56+00:00
4,p7370l7,datascience,CJDrew,1788222591,It varies but probably 40-50. Plus on call wee...,11,t3_1w3klb5,t1_p730wy3,2026-09-01 00:29:51+00:00


# 1.4 Saving results

In [6]:
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

posts_path = data_folder / f"{SUBREDDIT}_posts_2026-09.csv"
comments_path = data_folder / f"{SUBREDDIT}_comments_2026-09.csv"

posts.to_csv(posts_path, index=False)
comments.to_csv(comments_path, index=False)

print(f"Saved files in: {data_folder.resolve()}")

Saved files in: C:\Users\hej\OneDrive\Dokumenter\GitHub\CTDS_Group31\data


# 1.5 Call the CSV data after fetching

In later sessions, load the saved files directly so you do not need to call the API again:

In [ ]:
posts = pd.read_csv(posts_path, parse_dates=["created_at"])
comments = pd.read_csv(comments_path, parse_dates=["created_at"])